In [ ]:
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import accuracy_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
image_folder = r"C:\Users\rp122\Documents\8th sem\Datasets\steel-defect-detection\train_images"
csv_path = r"C:\Users\rp122\Documents\8th sem\Datasets\steel-defect-detection\train_full.csv"

# Load CSV
df = pd.read_csv(csv_path)

# Keep only rows with defects (EncodedPixels not null)
df = df[df["EncodedPixels"].notnull()]

# If multiple rows per image → keep first class
df = df.groupby("ImageId").first().reset_index()

class SteelDataset(Dataset):
    def __init__(self, dataframe, image_folder, processor):
        self.df = dataframe
        self.image_folder = image_folder
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["ImageId"]
        label = int(self.df.iloc[idx]["ClassId"]) - 1  # make 0-based

        img_path = os.path.join(self.image_folder, img_name)
        image = Image.open(img_path).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze()

        return pixel_values, torch.tensor(label)

# Load processor
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

dataset = SteelDataset(df, image_folder, processor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# Load model
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=4,
    ignore_mismatched_sizes=True
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

# Training
epochs = 5
model.train()

for epoch in range(epochs):
    all_preds = []
    all_labels = []
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {acc:.4f}")